# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JunaidAhamed-7777/flyrank-machine-learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Task type: ranking** (learning-to-rank / scoring).

Refresh Prediction for this capstone is not “predict decline yes/no.” Editors have finite weekly capacity; the business decision is **which pages to refresh first**, not whether a single URL is deteriorating. That decision needs an **ordered** output—a priority score per page and a queue sorted by expected traffic recovery if refreshed this week. A classifier can estimate components of risk, but the deliverable is a **rank**: the top *K* URLs an editor should touch before anything else. Classification answers a different question (membership in a decline bucket) and does not, by itself, allocate scarce hours across hundreds of in-scope pages per client.

## 2. Target or proxy

**Ideal target (not in the starter CSV):** measured **traffic recovered after a refresh** in a forward window—true recoverable impressions we would only see post-intervention.

**Proxy (observable now):** an **expected-recovery score** whose units are *recoverable impressions at stake this week*, built only from trailing-window fields we can inspect today. Intuitively: pages that look like they are **losing demand**, carry **enough impressions to matter**, and sit in a **position tier where CTR upside is still on the table** should score above pages that fail any leg of that tripod.

Plain English: *expected recovery rises when a page shows decline-like movement in recent impressions, when `impressions_90d` is large enough that a refresh could move meaningful clicks, and when `position_tier` / `ctr` imply headroom versus the SERP CTR cliff we measured in w01 (e.g. materially higher mean `ctr` in `top_3` than in `striking`).*

The raw file does **not** ship `is_declining_label`; the nearest observable decline flag is **`trend_direction`** (and its companion **`trend_pct`**), which the data dictionary defines from last-30d vs prev-30d impressions. We treat “in decline” for this proxy as **`trend_direction == "down"`**—a snapshot label for framing, not a claim of causal recovery.

```
expected_recovery_proxy
  ∝  decline_indicator(trend_direction)   # e.g. 1 when "down", else down-weighted
   × impressions_90d                      # exposure at stake
   × ctr_upside(position_tier, ctr)       # non-linear payoff from tier / current CTR
```
(Do not implement here—this notebook only fixes the contract.)

## 3. Success metric

**Primary:** **Precision@K** with **K = 50**, matching the starter pipeline’s headline comparison—of the 50 pages we rank highest for refresh this week, what fraction are truly high-priority under the held-out label (`is_declining_label` in the processed feature path / equivalent decline definition used in `scripts/03_train_model.py`).

**Secondary:** **NDCG@50**, rank-aware and recovery-weighted in spirit: it penalizes putting a high-exposure declining page at rank 40 while a low-stakes page sits at rank 5, even if both are technically “declining.”

**What “good” means:** beat the deterministic hand-written baseline from `scripts/02_baseline_score.py` on **Precision@50**, and report that gap under **client-holdout** validation (`scripts/03_train_model.py`—no client’s pages in both train and test). In-sample leaderboard numbers do not count; the honest claim is generalization to held-out clients, not memorizing one client’s quirks.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [1]:
import pandas as pd

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
PROXY_COLS = [
    "content_id",
    "client_id",
    "impressions_90d",
    "trend_direction",
    "trend_pct",
    "position_tier",
    "ctr",
    "avg_position",
]

df = pd.read_csv(DATA_PATH)
unit = df[PROXY_COLS].head(5)
unit


,content_id,client_id,impressions_90d,trend_direction,trend_pct,position_tier,ctr,avg_position
0,content_304f48230142,client_f369cb89fc,3803,down,-41.4,striking,0.76,10.6
1,content_a1fb4e703a9e,client_4e07408562,15320,down,-57.7,page_3_5,0.05,20.3
2,content_9aa793d4d895,client_7f2253d7e2,12581,down,-60.9,page_3_5,0.09,36.5
3,content_331d6c4de07b,client_19581e27de,11751,stable,-13.8,page_1,0.49,6.2
4,content_d99b7a2d90ca,client_3fdba35f04,19140,down,-34.7,page_3_5,0.13,44.0


**Unit of analysis:** One row = one page, scoped to one client (`content_id` × `client_id`).

## 5. Why ML beats a fixed rule here

- **CTR cliff × position tier:** In w01, mean `ctr` in `top_3` (~1.48% in this file’s units) dwarfed `striking` (~0.32%). The payoff of the same refresh action is **non-linear** in current tier—a global threshold on impressions or age cannot honestly weight “one step from page one” versus “deep” without learning that interaction.

- **Signal interactions, not single features:** Decline prevalence alone (~54% `trend_direction == down`) makes ordering the hard part. The starter pipeline’s random forest materially outperformed the hand rule on Precision@50 (~0.74 vs ~0.24 in `notebooks/01_first_look_and_discovery.ipynb`) because **joint patterns** across age, freshness, engagement, position, and CTR matter; w01 already showed univariate shortcuts (e.g. word count alone) do not carry the story.

- **Client-specific baselines:** A single global if-statement ignores that “what declining looks like” shifts by `client_id`. The repo’s **client-holdout** split exists because identity changes the feature distribution; rules baked at one threshold leak optimism on clients the rule never saw.

**What a model adds:** a **ranked** list whose ordering is fit to observed decline labels under an honest validation gap—something a fixed rule cannot recalibrate per client without becoming an unmaintainable nest of exceptions.

## 6. Self-check

- **Task type ranking, not classification?** Yes—the deliverable is an ordered refresh queue scored by expected recovery, not a binary decline detector.
- **Proxy observable in starter data?** Yes, using `trend_direction`, `trend_pct`, `impressions_90d`, `position_tier`, `ctr`, and `avg_position`; true post-refresh recovery is **not** in the CSV (ideal target deferred).
- **Metric named and comparable to an existing baseline?** Yes—Precision@50 vs `scripts/02_baseline_score.py`, evaluated like `scripts/03_train_model.py` (client-holdout).
- **Unit-of-analysis cell shows one row = one page?** Yes—`content_id` and `client_id` appear alongside proxy signal columns in the dataframe above.
- **Section 5 grounded in data, not vibes?** Yes—CTR tier gap, decline base rate, and documented baseline vs model Precision@50 from the starter pipeline.

Submission hygiene:

- [x] Notebook runs top to bottom (section 4 code only)
- [x] No client names, URLs, or private queries
- [x] Claims use decision-support / directional language where appropriate
- [x] Committed under `work/notebooks/` and pushed to `main`